# v73 — GONG SSL → refiner encoder init (fold2 screen) — Colab 版

Kaggle kernel `solar-filament-v73-ssl-refiner` と同一実験。
過去の Colab notebook と同じく **Drive が主データソース**:

- `solar_filament_v30/` から train json・test_images・preproc・folds CSV
  （sha 検証済み）・baseline/YOLO ckpt を直読み
- kaggle API は corpus (`solar-gong-ssl`)・code (`solar-v73-code`)・
  reviewed manifest (`solar-v73-assets` 内1ファイル) のみ
- kaggle 認証は Drive `solar_filament_v35/kaggle.json`（shingo257）

**Provenance pin:** supply V73_PROVENANCE_SHA256 only after independent review
of actual checkpoint-to-training-split evidence. The candidate manifest in
solar-v73-assets has not passed that review. A matching Dice/OOF score is
insufficient; see docs/V73_PROVENANCE.md.


In [ ]:
import os, re, json, subprocess, sys, hashlib
from pathlib import Path
# Reviewed provenance pin is supplied externally — env var, Colab secret,
# Kaggle secret — or declared interactively. Never derived from artifacts.
EXPECTED_PROVENANCE_SHA = os.environ.get('V73_PROVENANCE_SHA256', '')
if not EXPECTED_PROVENANCE_SHA:
    try:
        from google.colab import userdata
        EXPECTED_PROVENANCE_SHA = userdata.get('V73_PROVENANCE_SHA256') or ''
    except Exception:
        pass
if not EXPECTED_PROVENANCE_SHA:
    try:
        from kaggle_secrets import UserSecretsClient
        EXPECTED_PROVENANCE_SHA = UserSecretsClient().get_secret('V73_PROVENANCE_SHA256')
    except Exception:
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    try:
        EXPECTED_PROVENANCE_SHA = input('V73_PROVENANCE_SHA256 (reviewed manifest hash): ').strip()
    except (EOFError, OSError):
        pass
if not re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PROVENANCE_SHA):
    raise RuntimeError('Set V73_PROVENANCE_SHA256 after independent provenance review')

# [0] Drive mount — primary data source (same pattern as all past colab runs)
from google.colab import drive
drive.mount('/content/drive')
V30 = Path('/content/drive/MyDrive/solar_filament_v30')
COMP = V30 / 'competition_data/MAGFiLO_1.0_Kaggle_2026'
inputs = {
    'train_json': COMP / 'train/MAGFiLO_1.0_Annotations_kaggle2026_train.json',
    'test_dir': COMP / 'test/test_images',
    'preproc_dir': V30 / 'preprocessed_v30_q96',
    'folds_csv': V30 / 'artifacts/grouped_folds.csv',
    'baseline_ckpt': V30 / 'artifacts/mediseg_edgeattnet_refiner_fold2_resume.pt',
    'yolo_ckpt': V30 / 'runs/yolo_fold_2/weights/best.pt',
    'kaggle.json': Path('/content/drive/MyDrive/solar_filament_v35/kaggle.json'),
}
missing = {k: str(v) for k, v in inputs.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f'Drive inputs missing: {missing}')
train_json, test_dir, preproc_dir = (inputs['train_json'], inputs['test_dir'],
                                     inputs['preproc_dir'])
folds_csv, baseline_ckpt, yolo_ckpt = (inputs['folds_csv'], inputs['baseline_ckpt'],
                                      inputs['yolo_ckpt'])
if not any(test_dir.iterdir()):
    raise RuntimeError('test_images dir is empty')
EXPECTED_FOLDS_SHA = 'febb3f67237a058fa8a71f188fac262035048b55d860c0e4509514877f6af67e'
actual = hashlib.sha256(folds_csv.read_bytes()).hexdigest()
if actual != EXPECTED_FOLDS_SHA:
    raise RuntimeError(f'fold CSV sha mismatch: {actual} (Drive artifacts/grouped_folds.csv)')
print('folds_csv sha256 OK (true experimental split)')

# [1] Kaggle auth (Drive kaggle.json -> Secrets -> upload) for corpus/code/manifest
os.makedirs('/root/.kaggle', exist_ok=True)
cred = inputs['kaggle.json'].read_bytes()
print('auth: Drive kaggle.json')
try:
    _u = json.loads(cred)['username']
except Exception:
    _u = '?'
print('kaggle user:', _u)
Path('/root/.kaggle/kaggle.json').write_bytes(cred)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                'kaggle', 'ultralytics', 'pycocotools'], check=True)

WORK = Path('/content/work'); IN = WORK / 'input'; IN.mkdir(parents=True, exist_ok=True)
def _dl(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('download failed rc=%s: %s\n%s\n%s' % (
            r.returncode, cmd, r.stdout[-800:], r.stderr[-800:]))
# Only what Drive cannot provide: SSL corpus, code, reviewed manifest file.
_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-code',
     '-p', str(IN), '--unzip', '-q'])
_dl(['kaggle', 'datasets', 'download', 'shingo257/solar-v73-assets',
     '-f', 'v73_reviewed_provenance.json', '-p', str(IN)])
# solar-gong-ssl: kaggle's whole-archive DownloadDataset endpoint 404s for this
# dataset despite Ready status; fetch the fixed v3 file set individually.
GONG = IN / 'gong_ssl'
GONG.mkdir(exist_ok=True)
for _f in ['batch_000.npz', 'batch_001.npz', 'batch_002.npz', 'batch_003.npz',
           'batch_004.npz', 'batch_005.npz', 'batch_006.npz', 'manifest.jsonl',
           'corpus_config.json', 'corpus_summary.json', 'upgrade_dropped_files.json']:
    _dl(['kaggle', 'datasets', 'download', 'shingo257/solar-gong-ssl',
         '-f', _f, '-p', str(GONG)])

corpus_dir = next(p.parent for p in IN.rglob('batch_*.npz'))
code_dir = next(p.parent for p in IN.rglob('train_refiner_fold.py'))
sys.path.insert(0, str(code_dir))
# Provenance is a reviewed input, never auto-generated from ambiguous legacy weights.
manifest_path = next(IN.rglob('v73_reviewed_provenance.json'))
from v73_split_provenance import (load_reviewed_manifest, load_fold_mapping,
                                  validate_checkpoint_binding)
manifest = load_reviewed_manifest(manifest_path, EXPECTED_PROVENANCE_SHA, 2)
if manifest['folds_sha256'] != EXPECTED_FOLDS_SHA:
    raise RuntimeError('reviewed split differs')
coco = json.loads(train_json.read_text())
load_fold_mapping(folds_csv, {i['file_name'] for i in coco['images']}, 2,
                  manifest['folds_sha256'])
validate_checkpoint_binding(yolo_ckpt, 'yolo', manifest)
validate_checkpoint_binding(baseline_ckpt, 'baseline', manifest)
# No package installs, CUDA inspection or training until all artifact preflights pass.
subprocess.run([sys.executable, str(code_dir / 'pretrain_unet_encoder_ssl.py'),
                '--corpus', str(corpus_dir), '--train-json', str(train_json),
                '--test-dir', str(test_dir), '--validate-only',
                '--out', str(WORK / 'unet_encoder_ssl.pt')], check=True)
for name, p in [('corpus', corpus_dir), ('preproc', preproc_dir),
                ('folds', folds_csv), ('baseline', baseline_ckpt),
                ('yolo', yolo_ckpt), ('train_json', train_json),
                ('test_dir', test_dir), ('code', code_dir)]:
    print(name, '->', p)
import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
%%time
# [1/4] GONG corpus で U-Net encoder を SimCLR 事前学習 (solar3 変換込み)
gpu_env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
r = subprocess.run([sys.executable,
                    str(code_dir / 'pretrain_unet_encoder_ssl.py'),
                    '--corpus', str(corpus_dir),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--epochs', '60', '--batch', '32', '--crop', '320',
                    '--out', str(WORK / 'unet_encoder_ssl.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'unet_encoder_ssl.pt').exists():
    raise RuntimeError(f'SSL pretrain failed rc={r.returncode}')


In [ ]:
%%time
# [2/4] fold2 refiner 学習 (folds 0+1 で学習、fold2 valid) — scratch arm
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt', '--init', 'scratch',
                    '--epochs', '14', '--seed', '3407',
                    '--out', str(WORK / 'edgeatt_scratch_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_scratch_fold2.pt').exists():
    raise RuntimeError(f'scratch arm failed rc={r.returncode}')


In [ ]:
%%time
# [3/4] 同一レシピ・同一 seed — ssl encoder init arm
if not (WORK / 'unet_encoder_ssl.pt').exists():
    raise RuntimeError('SSL encoder checkpoint missing — pretrain cell failed')
r = subprocess.run([sys.executable, str(code_dir / 'train_refiner_fold.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--expected-folds-sha256', manifest['folds_sha256'],
                    '--model', 'edgeatt',
                    '--init', f'ssl:{WORK}/unet_encoder_ssl.pt',
                    '--epochs', '14', '--seed', '3407',
                    '--out', str(WORK / 'edgeatt_ssl_fold2.pt')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-3000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'edgeatt_ssl_fold2.pt').exists():
    raise RuntimeError(f'ssl arm failed rc={r.returncode}')


In [ ]:
%%time
# [4/4] fold2 OOF eval — 3 arm を同一 YOLO 提案・同一後処理で採点
for _p in ('edgeatt_scratch_fold2.pt', 'edgeatt_ssl_fold2.pt'):
    if not (WORK / _p).exists():
        raise RuntimeError(f'refiner checkpoint missing: {_p}')
r = subprocess.run([sys.executable, str(code_dir / 'run_fold_oof_eval.py'),
                    '--train-json', str(train_json),
                    '--test-dir', str(test_dir),
                    '--preproc-dir', str(preproc_dir),
                    '--folds-csv', str(folds_csv), '--fold', '2',
                    '--provenance-manifest', str(manifest_path),
                    '--provenance-sha256', EXPECTED_PROVENANCE_SHA,
                    '--yolo', str(yolo_ckpt), '--model', 'edgeatt',
                    '--refiner', f'baseline={baseline_ckpt}',
                    '--refiner', f'scratch={WORK}/edgeatt_scratch_fold2.pt',
                    '--refiner', f'ssl={WORK}/edgeatt_ssl_fold2.pt',
                    '--out', str(WORK / 'v73_eval.json')],
                   capture_output=True, text=True, env=gpu_env)
print(r.stdout[-4000:]); print('STDERR:', r.stderr[-2000:])
if r.returncode != 0 or not (WORK / 'v73_eval.json').exists():
    raise RuntimeError(f'eval failed rc={r.returncode}')


In [ ]:
# 結果サマリ — 事前固定ゲート判定
rep = json.load(open(WORK / 'v73_eval.json'))
arms = rep['arms']
table = {a: r['pq_by_threshold'] for a, r in arms.items()}
print(json.dumps(table, indent=2))
gate = {
  'baseline_pq_0.45': arms['baseline']['pq_by_threshold']['0.45'],
  'scratch_pq_0.45': arms['scratch']['pq_by_threshold']['0.45'],
  'ssl_pq_0.45': arms['ssl']['pq_by_threshold']['0.45'],
  'delta_ssl_minus_scratch': round(
      arms['ssl']['pq_by_threshold']['0.45']
      - arms['scratch']['pq_by_threshold']['0.45'], 6),
  'sanity_baseline_near_0.448':
      abs(arms['baseline']['pq_by_threshold']['0.45'] - 0.4483) < 0.01,
}
print(json.dumps(gate, indent=2))
Path(WORK / 'v73_summary.json').write_text(json.dumps(
    {'eval': rep, 'gate': gate}, indent=2))


In [ ]:
# [optional] Mirror work artifacts to a mounted Drive and verify copied bytes.
from google.colab import drive
import hashlib, os, shutil, time
DRIVE_ROOT = Path('/content/drive')
drive.mount(str(DRIVE_ROOT))

def require_drive():
    if not os.path.ismount(DRIVE_ROOT) or not (DRIVE_ROOT / 'MyDrive').is_dir():
        raise RuntimeError('Drive is not mounted; durable mirror unavailable')

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

require_drive()
if not WORK.is_dir():
    raise RuntimeError('Work directory is missing')
OUT = DRIVE_ROOT / 'MyDrive/solar_filament_v73/work' / time.strftime('%Y%m%d_%H%M%S')
OUT.mkdir(parents=True, exist_ok=False)
digests = {}
for p in sorted(WORK.rglob('*')):
    if p.is_file():
        require_drive()
        dst = OUT / p.relative_to(WORK)
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dst)
        expected = file_sha256(p)
        if file_sha256(dst) != expected:
            raise RuntimeError(f'Drive mirror SHA256 mismatch: {p.name}')
        digests[str(p.relative_to(WORK))] = expected
        print('mirrored', p.name, p.stat().st_size)
if not digests:
    raise RuntimeError('No work artifacts to mirror')
require_drive()
(OUT / 'sha256_manifest.json').write_text(json.dumps(digests, indent=1))
require_drive()
print('DONE ->', OUT)
